# Dev notebook for generic volume data

## Imports

In [ ]:
%load_ext line_profiler
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
from typing import Tuple
from matplotlib import pyplot as plt
from skimage import measure
from PIL import Image
import numpy.lib.recfunctions as rf
import os
os.add_dll_directory(r"C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v12.9\bin")
from _preprocess_module import HeightFieldExtractor
from smooth_generic_volume_container import SmoothGenericVolumeContainer
from generic_volume_container import GenericVolumeContainer
import create_volume


## Python class to represent generic volume data

In [ ]:
PRINT_COORD_VALS = False
PLOT_IMG = False

def visualize_container(volume: GenericVolumeContainer) -> None:
    fig = plt.figure()
    ax = fig.add_subplot(projection="3d")
    ax.voxels(volume.container)
    ax.set_xlim(0, volume.size_x)
    ax.set_ylim(0, volume.size_y)
    ax.set_zlim(0, volume.size_z)
    plt.show()

def print_volume_values_running_z(volume: GenericVolumeContainer, x: int, y: int, z_start: int, z_stop:int) -> None:
    print(f"Values for x: {x}, y: {y}, running z from {z_start} to {z_stop}")
    for i in range(z_start, z_stop):
        print(f"Value at z {i}: {volume.get_value_at_coord((x,y,i))}")

def test_generic_volume():
    volume = GenericVolumeContainer((50,50,50))
    volume.add_cuboid((10,10,10), (20,20,20), (45,20,20))
    volume.add_sphere(10, (10,10,10))

    volume.add_cylinder(10, 30, (30,30,10), (10,10,45))
    if PRINT_COORD_VALS:
        print_volume_values_running_z(volume, 19, 10, 0, 20)
    if PLOT_IMG:
        visualize_container(volume)
    del volume
    

In [ ]:
test_generic_volume()

## Interfaces needed to transfer the volume data to cpp

### add_volume

- [x] shape information to allocate a matrix
- [x] copy the matrix

### intersect
- [x] Extract the extended heighfield using texture memory
- [ ] Extract the normal map

## Notes

- Result of the heightfield is faulty. The result is copied 4 times in the image.

In [ ]:
def create_py_volume_huge_and_proc():
    vol = SmoothGenericVolumeContainer((1000,900,500))
    vol.add_cuboid((120,120,120), (40,40,20), (45,45,45)) # x, y, z
    vol.add_cuboid((50, 50, 50), (50,50,160))
    vol.add_sphere(50, (800, 800, 300))
    vol.add_sphere(40, (800, 800, 400))
    vol.add_cylinder(30, 200, (400,400,400), (10,10,45))
    proc = HeightFieldExtractor((900,1000), 4, 256)
    # proc.add_volume(vol.container, (1000,900,500), 0.001)
    return (vol,proc)

def create_py_volume_small_and_proc():
    vol = GenericVolumeContainer((128,128,128))
    vol.add_sphere(40, (50,50,50))
    proc = HeightFieldExtractor((128,128), 4, 256)
    proc.add_volume(vol.container, (128,128,128), 0.0625)
    return (vol, proc)
# vol = create_py_volume()
# preprocessor = HeightFieldExtractor( (900, 1000), 2, 256 )
# preprocessor.add_volume(vol.container, (1000, 900, 500))


def test_implementation():
    # vol, preprocessor = create_py_volume_huge_and_proc()

    vol = create_volume.create_huge_volume()
    preprocessor = HeightFieldExtractor((create_volume.RESOLUTION_Y, create_volume.RESOLUTION_X), 4, 256)
    preprocessor.add_volume(vol.container, (create_volume.RESOLUTION_X, create_volume.RESOLUTION_Y, create_volume.RESOLUTION_Z), 0.001)

    extended_heightfield, normal_map = preprocessor.extract_data_representation( 0.0 )

    # Save the extended heightfields
    for z in range(extended_heightfield.shape[2]):
        # entry_0 = rf.structured_to_unstructured(extended_heightfield[:,:,z]);
        entry = extended_heightfield[:,:,z]
        entry = entry.astype(np.uint16)
        img = Image.fromarray(entry, "I;16")
        img.save("output/integrated_"+str(z)+".tif")

    # Convert the first normal map
    normal_map = normal_map.squeeze(2)
    normal_map = rf.structured_to_unstructured( normal_map )
    normal_map = ( normal_map + 1.0 ) * 127.5
    normal = normal_map.astype(np.uint8)
    # print(normal)
    img = Image.fromarray(normal, "RGB")
    img.save("output/normal.tif")
    del preprocessor
    del vol

In [ ]:
%lprun -f test_implementation test_implementation()
# test_implementation()

In [ ]:
def create_py_volume_for_file() -> GenericVolumeContainer:
    vol = GenericVolumeContainer((10,10,10))
    vol.add_sphere(5, (5, 5, 5))
    with open("output/volume_data.txt", "w") as f:
        f.write(str(vol.container))
    return vol


def write_img_to_file():

    vol = create_py_volume_for_file()
    pp = HeightFieldExtractor( (10, 10), 2, 256 )
    pp.add_volume(vol.container, (10, 10, 10), 0.0625)


    extended_heightfield, normal_map = pp.extract_data_representation( 0.0 )

    # Save the extended heightfields
    for z in range(extended_heightfield.shape[2]):
        # entry_0 = rf.structured_to_unstructured(extended_heightfield[:,:,z]);
        entry = extended_heightfield[:,:,z]
        entry = entry.astype(np.uint16)
        img = Image.fromarray(entry, "I;16")
        img.save("output/integrated_"+str(z)+".tif")

    # Convert the first normal map
    normal_map = normal_map.squeeze(2)
    normal_map = rf.structured_to_unstructured( normal_map )
    normal_map = ( normal_map + 1.0 ) * 127.5
    normal = normal_map.astype(np.uint8)
    # print(normal)
    # Write data to file
    with open("output/vol_normal.txt", "w") as f:
        f.write(str(normal))
    img = Image.fromarray(normal, "RGB")
    img.save("output/normal.tif")
    del pp
    del vol

In [ ]:
# write_img_to_file()